## Úvod 

Táto lekcia pokryje: 
- Čo je volanie funkcie a jej použitia 
- Ako vytvoriť volanie funkcie pomocou Azure OpenAI 
- Ako integrovať volanie funkcie do aplikácie 

## Ciele učenia 

Po dokončení tejto lekcie budete vedieť a rozumieť: 

- Účel použitia volania funkcií 
- Nastavenie volania funkcie pomocou Azure Open AI Service 
- Navrhnite efektívne volania funkcií pre použitie vo vašej aplikácii 


## Pochopenie volaní funkcií

Pre túto lekciu chceme vytvoriť funkciu pre náš vzdelávací startup, ktorá umožní používateľom použiť chatbot na vyhľadávanie technických kurzov. Budeme odporúčať kurzy, ktoré zodpovedajú ich úrovni zručností, aktuálnej pozícii a záujmom o technológie.

Na dokončenie použijeme kombináciu:
 - `Azure Open AI` na vytvorenie chatovacieho zážitku pre používateľa
 - `Microsoft Learn Catalog API` na pomoc používateľom nájsť kurzy podľa ich požiadavky
 - `Function Calling`, aby sme zobrali dotaz používateľa a poslali ho do funkcie na vykonanie API požiadavky.

Na začiatok sa pozrime, prečo by sme chceli vôbec použiť volanie funkcií:


### Prečo volanie funkcií

Ak ste absolvovali akúkoľvek inú lekciu v tomto kurze, pravdepodobne rozumiete sile používania veľkých jazykových modelov (LLM). Dúfajme, že tiež vidíte niektoré z ich obmedzení.

Volanie funkcií je funkcia služby Azure Open AI, ktorá prekonáva nasledujúce obmedzenia:
1) Konzistentný formát odpovede
2) Schopnosť použiť údaje z iných zdrojov aplikácie v kontexte chatu

Pred volaním funkcií boli odpovede z LLM neštruktúrované a nekonzistentné. Vývojári museli písať zložitý validačný kód, aby dokázali spracovať každú variantu odpovede.

Používatelia nedokázali získať odpovede ako „Aké je aktuálne počasie v Štokholme?“. Je to preto, že modely boli obmedzené na čas, kedy boli dáta vytrénované.

Pozrime sa na príklad nižšie, ktorý ilustruje tento problém:

Predstavme si, že chceme vytvoriť databázu údajov o študentoch, aby sme im mohli navrhnúť správny kurz. Nižšie máme dva popisy študentov, ktoré sú veľmi podobné v údajoch, ktoré obsahujú.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Chceme to odoslať do LLM, aby spracoval údaje. To môže byť neskôr použité v našej aplikácii na odoslanie do API alebo uloženie do databázy.

Vytvorme dve identické výzvy, ktorými inštruujeme LLM, o ktoré informácie máme záujem:


Chceme to poslať LLM, aby rozpoznal časti, ktoré sú dôležité pre náš produkt. Môžeme teda vytvoriť dva identické prompt-y na inštruovanie LLM: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Po vytvorení týchto dvoch promptov ich odošleme do LLM pomocou `client.responses.create`.  Prompt uložíme do premennej `input` a priradíme rolu `user`. Toto má napodobniť správu od používateľa, ktorá sa píše do chatbota. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Teraz môžeme odoslať obe požiadavky do LLM a preskúmať odpoveď, ktorú dostaneme. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Aj keď sú výzvy rovnaké a popisy podobné, môžeme získať rôzne formáty vlastnosti `Grades`.

Ak spustíte vyššie uvedenú bunku viackrát, formát môže byť `3.7` alebo `3.7 GPA`.

Je to preto, že LLM spracováva nestruktúrované údaje vo forme napísanej výzvy a vracia tiež nestruktúrované údaje. Potrebujeme mať štruktúrovaný formát, aby sme vedeli, čo očakávať pri ukladaní alebo používaní týchto údajov.

Použitím funkčného volania môžeme zabezpečiť, že dostaneme späť štruktúrované údaje. Pri používaní funkčného volania LLM skutočne nevolá ani nespúšťa žiadne funkcie. Namiesto toho vytvoríme štruktúru, ktorú má LLM dodržiavať pre svoje odpovede. Potom tieto štruktúrované odpovede použijeme na určenie, ktorú funkciu spustiť v našich aplikáciách.
 


![Diagram toku volania funkcie](../../../../translated_images/sk/Function-Flow.083875364af4f4bb.webp)


Následne môžeme vziať to, čo funkcia vráti, a poslať to späť do LLM. LLM potom odpovie pomocou prirodzeného jazyka, aby zodpovedal používateľovu otázku. 


### Použitie volaní funkcií 

**Volanie externých nástrojov** 
Chatboty sú skvelé na poskytovanie odpovedí na otázky používateľov. Použitím volania funkcií môžu chatboty využiť správy od používateľov na vykonanie určitých úloh. Napríklad študent môže požiadať chatbota, aby „poslal email môjmu lektorovi s tým, že potrebujem viac pomoci s týmto predmetom“. Tento prípad môže vyvolať volanie funkcie `send_email(to: string, body: string)`


**Tvorba API alebo databázových dopytov**
Používatelia môžu nájsť informácie pomocou prirodzeného jazyka, ktorý sa konvertuje na formátovaný dopyt alebo API požiadavku. Príkladom môže byť učiteľ, ktorý požaduje „Kto sú študenti, ktorí dokončili poslednú úlohu“, čo môže vyvolať volanie funkcie `get_completed(student_name: string, assignment: int, current_status: string)`


**Tvorba štruktúrovaných dát**
Používatelia môžu zobrať blok textu alebo CSV a použiť LLM na extrahovanie dôležitých informácií z neho. Napríklad študent môže konvertovať článok z Wikipédie o mierových dohodách, aby vytvoril AI kartičky na učenie. Tento proces môže byť vykonaný pomocou funkcie s názvom `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Vytvorenie vášho prvého volania funkcie 

Proces vytvorenia volania funkcie zahŕňa 3 hlavné kroky: 
1. Volanie rozhrania API Chat Completions s zoznamom vašich funkcií a správou od používateľa 
2. Prečítanie odpovede modelu na vykonanie akcie, t.j. spustenie funkcie alebo volania API 
3. Vykonajte ďalšie volanie rozhrania API Chat Completions s odpoveďou z vašej funkcie, aby ste mohli použiť tieto informácie na vytvorenie odpovede pre používateľa. 


![Priebeh volania funkcie](../../../../translated_images/sk/LLM-Flow.3285ed8caf4796d7.webp)


### Prvky volania funkcie 

#### Vstup používateľa 

Prvým krokom je vytvoriť správu používateľa. Tá môže byť dynamicky priradená odobratím hodnoty z textového vstupu alebo môžete hodnotu priradiť priamo tu. Ak pracujete s API Chat Completions prvýkrát, je potrebné definovať `role` a `content` správy. 

`role` môže byť `system` (vytváranie pravidiel), `assistant` (model) alebo `user` (koncový používateľ). Pre volanie funkcie túto hodnotu nastavíme ako `user` spolu s príkladovou otázkou. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Vytváranie funkcií. 

Ďalej definujeme funkciu a jej parametre. Použijeme tu iba jednu funkciu nazvanú `search_courses`, ale môžete vytvoriť viacero funkcií.

**Dôležité** : Funkcie sú zahrnuté v systémovej správe pre LLM a budú započítané do množstva dostupných tokenov, ktoré máte k dispozícii. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definície** 

`name` - Meno funkcie, ktorú chceme volať. 

`description` - Toto je popis, ako funkcia funguje. Tu je dôležité byť konkrétny a jasný 

`parameters` - Zoznam hodnôt a formátu, ktoré chcete, aby model vytvoril vo svojej odpovedi 


`type` - Dátový typ, v ktorom budú uložené vlastnosti 

`properties` - Zoznam konkrétnych hodnôt, ktoré model použije vo svojej odpovedi 


`name` - meno vlastnosti, ktorú model použije vo svojej formátovanej odpovedi 

`type` - Dátový typ tejto vlastnosti 

`description` - Popis konkrétnej vlastnosti 


**Voliteľné**

`required` - požadovaná vlastnosť pre dokončenie volania funkcie 


### Volanie funkcie 
Po definovaní funkcie ju teraz musíme zahrnúť do volania API Chat Completion. Robíme to tak, že do požiadavky pridáme `functions`. V tomto prípade `functions=functions`. 

Existuje aj možnosť nastaviť `function_call` na `auto`. To znamená, že necháme LLM rozhodnúť, ktorá funkcia by sa mala volať na základe správy používateľa, namiesto toho, aby sme ju priraďovali sami.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Teraz sa pozrime na odpoveď a uvidíme, ako je naformátovaná: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Vidíte, že je zavolaná funkcia podľa jej mena a z používateľskej správy dokázal LLM nájsť údaje vyhovujúce argumentom funkcie. 


## 3. Integrácia volaní funkcií do aplikácie. 


Po tom, čo sme otestovali formátovanú odpoveď od LLM, ju teraz môžeme integrovať do aplikácie. 

### Riadenie toku 

Na integráciu do našej aplikácie urobme nasledovné kroky: 

Najskôr vykonajme volanie na služby Open AI a uložme správu do premennej volanej `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Teraz definujeme funkciu, ktorá zavolá Microsoft Learn API, aby získala zoznam kurzov: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Ako dobrú prax si potom overíme, či model chce zavolať funkciu. Následne vytvoríme jednu z dostupných funkcií a priradíme ju k funkcii, ktorá sa má zavolať.
Potom vezmeme argumenty funkcie a namapujeme ich na argumenty z LLM.

Nakoniec pridáme správu o volaní funkcie a hodnoty, ktoré boli vrátené správou `search_courses`. Týmto spôsobom dostane LLM všetky informácie, ktoré potrebuje
na odpoveď používateľovi pomocou prirodzeného jazyka.


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Teraz odošleme aktualizovanú správu do LLM, aby sme mohli dostať odpoveď v prirodzenom jazyku namiesto odpovede vo formáte API JSON. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Kódová výzva 

Skvelá práca! Ak chcete pokračovať v učení o Azure Open AI Function Calling, môžete vytvoriť: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Viac parametrov funkcie, ktoré môžu pomôcť študentom nájsť viac kurzov. Dostupné API parametre nájdete tu: 
 - Vytvorte ďalší volanie funkcie, ktoré zoberie viac informácií od študenta, napríklad ich rodný jazyk 
 - Vytvorte spracovanie chýb, keď volanie funkcie a/alebo API volanie nevrátia žiadne vhodné kurzy 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
